In [46]:
import requests
import numpy as np
from skyfield.api import EarthSatellite, load, wgs84, utc
import datetime as dt

In [4]:
tlesource = requests.get("https://celestrak.org/NORAD/elements/gp.php?GROUP=active").text.strip().split("\n")
tles = [tlesource[i:i+3] for i in range(0, len(tlesource), 3)]
print(f"Loaded {len(tles)} TLEs")

def get_tle(sat_name):
    for tle in tles:
        if tle[0].strip() == sat_name:
            return tle
    return None

Loaded 14121 TLEs


In [ ]:
def satellite_elevation(t, lat_deg, lon_deg, height_m, tle_3linelist):
    ts = load.timescale()
    t_py = dt.datetime.fromtimestamp(t.astype('datetime64[us]').astype('int') / 1e6, utc).replace(tzinfo=utc)
    t_sf = ts.from_datetime(t_py)

    station = wgs84.latlon(
        latitude_degrees=lat_deg,
        longitude_degrees=lon_deg,
        elevation_m=height_m,
    )
    observer_gcrs = station.at(t_sf)

    def runone(tle_3line, ts, t_sf, observer_gcrs):
        name, line1, line2 = tle_3line
        satellite = EarthSatellite(line1, line2, name, ts)
        sat_gcrs = satellite.at(t_sf)
        difference = sat_gcrs - observer_gcrs
        alt, _, _ = difference.altaz()
        return alt.degrees
    
    for tle in tle_3linelist:
        alt_deg = runone(tle, ts, t_sf, observer_gcrs)
    return True

In [65]:
# Moscow:
lat_moscow = 55.7
lon_moscow = 37.1
height_moscow = 170
t = np.datetime64('now')


In [67]:
import time
start_time = time.perf_counter()
satellite_elevation(t, lat_moscow, lon_moscow, height_moscow, tles)
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"{len(tles)} satellites in {elapsed_time:.4f} sec")
average_time = elapsed_time / len(tles)
print(f"Average time per satellite: {average_time:.6f} seconds.")

14121 satellites in 1.3787 sec
Average time per satellite: 0.000098 seconds.
